In [1]:
# import packages
from __future__ import annotations

import math
import os
import sys
import time
from pathlib import Path
from typing import Tuple, Union
import threading
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from catboost import CatBoostClassifier
from classifier_calibration.calibration_error import classwise_ece
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import  log_loss
from sklearn.metrics import precision_score, recall_score

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
np.random.seed(42)

In [4]:
# import custom functions
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Thesis code" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from Functions.data_utils import (
    plot_incremental_response_rate,
    uplift_by_decile_bin,
    coerce_metrics_to_numeric,
)

Failed to import duecredit due to No module named 'duecredit'


In [5]:
file_path = r"Data/covariates_modeling_uplift_models_2026-03-13.csv"
df = pd.read_csv(file_path)

C:\Users\tsterk\AppData\Local\Temp\ipykernel_22792\3407484870.py:2: DtypeWarning: Columns (0: monetary_value, 1: total_volume, 2: online_sales, 3: retail_sales, 4: food_total, 5: vhms_total, 6: sports_total, 7: beauty_total, 8: monetary_value_52wk, 9: online_sales_52w, 10: retail_sales_52w, 11: monetary_value_53w_104w, 12: online_sales_53w_104w, 13: retail_sales_53w_104w) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [6]:
df['reactivated'].value_counts(normalize =True)

reactivated
0    0.978221
1    0.021779
Name: proportion, dtype: float64

In [7]:
# drop not performing incentives
drop_indicators = [
    "BNLX_ChurnP_SKUd_test_export.csv",
    "BNLX_ChurnP_SKUd_controle_export.csv",
    "BNLX_ChurnP_niks_test_export.csv",
    "BNLX_ChurnP_niks_controle_export.csv",
]

df = df[~df["treatment_indicator"].isin(drop_indicators)]

In [8]:
# convert for easy processing
treatment_converter = {
    "BNLX_ChurnP_10_test_export.csv": 1,
    "BNLX_ChurnP_10_controle_export.csv": 0, 
    
    "BNLX_ChurnP_25_test_export.csv": 2,
    "BNLX_ChurnP_25_controle_export.csv": 0,

    "BNLX_ChurnP_5eu_test_export.csv": 3,
    "BNLX_ChurnP_5eu_controle_export.csv": 0,

    "BNLX_ChurnP_10eu_test_export.csv": 4,
    "BNLX_ChurnP_10eu_controle_export.csv": 0,
    
    "BNLX_ChurnP_250_test_export.csv": 5,
    "BNLX_ChurnP_250_controle_export.csv": 0,

    "BNLX_ChurnP_500_test_export.csv": 6,
    "BNLX_ChurnP_500_controle_export.csv": 0,    

    "BNLX_ChurnP_SKUe_test_export.csv": 7,
    "BNLX_ChurnP_SKUe_controle_export.csv": 0,    

}

df["treatment"] = df["treatment_indicator"].map(treatment_converter)


In [9]:
categorical_cols  = ['has_rfl','gender','country_sk']
numeric_cols = [ 'recency', 'frequency', 'monetary_value',  'total_volume', 'length_of_relationship', 'online_sales', 'retail_sales', 
      'food_total', 'vhms_total', 'sports_total', 'beauty_total', 'frequency_52wk', 'monetary_value_52wk', 'volume_52wk', 'online_sales_52w', 
       'retail_sales_52w', 'frequency_53w_104w', 'monetary_value_53w_104w', 'volume_53w_104w', 'online_sales_53w_104w','retail_sales_53w_104w']

In [10]:
df = coerce_metrics_to_numeric(df, numeric_cols)
df[categorical_cols] = df[categorical_cols].astype("object")
df[numeric_cols] = df[numeric_cols].astype("int64")

In [11]:
# create dependent multi outcome variable
df["multi_outcome"] = np.where(
    df["reactivated"].to_numpy() != 0,
    "reactivated_" + df["treatment"].astype(str),
    "no_reactivated_" + df["treatment"].astype(str),
)

y = df["multi_outcome"]
t = df["treatment"]
X = df.drop(columns=["multi_outcome", "treatment_indicator", 'treatment', "reactivated", 'customer_nk'])

In [12]:
df['multi_outcome'].value_counts(normalize = True)

multi_outcome
no_reactivated_0    0.491382
no_reactivated_5    0.071349
no_reactivated_2    0.071140
no_reactivated_4    0.069462
no_reactivated_6    0.069414
no_reactivated_1    0.068732
no_reactivated_3    0.068580
no_reactivated_7    0.068030
reactivated_0       0.010313
reactivated_6       0.001797
reactivated_3       0.001735
reactivated_5       0.001726
reactivated_4       0.001674
reactivated_2       0.001598
reactivated_7       0.001550
reactivated_1       0.001517
Name: proportion, dtype: float64

In [13]:
# Fill missing values
def fill_cat_missing(df, cat_cols):
    df = df.copy()
    df[cat_cols] = df[cat_cols].astype("string").fillna("MISSING")
    return df

In [14]:
# monitoring function
def _start_fit_watchdog(label: str, every_s: int = 600):
    stop_event = threading.Event()
    t0 = time.perf_counter()

    def _worker():
        next_print = t0 + every_s
        while not stop_event.is_set():
            remaining = next_print - time.perf_counter()
            if remaining > 0:
                stop_event.wait(remaining)
            if stop_event.is_set():
                break

            elapsed = time.perf_counter() - t0
            print(f"[Dirichlet:{label}] still fitting (elapsed={elapsed/600:.1f} min)")
            next_print += every_s

    thread = threading.Thread(target=_worker, daemon=True)
    thread.start()
    return stop_event, t0


def _stop_fit_watchdog(stop_event, t0, label: str):
    stop_event.set()
    elapsed = time.perf_counter() - t0
    print(f"[Dirichlet:{label}] fit finished (elapsed={elapsed/600:.2f} min)")

In [15]:
y_series = pd.Series(y, index=X.index)
all_classes = np.unique(y_series)

# ── Cross-validation setup ─────────────────────────────────────────────────────
# Stratify on treatment × outcome to ensure each fold has balanced class/treatment
# representation across both the uplift labels and target variable
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_s = pd.Series(y, index=X.index).astype(str)
t_s = pd.Series(t, index=X.index).astype(str)
strata = t_s + "_" + y_s.astype(str)

# output holders predictions
fold_results_uncal = []
fold_results_cal = []
fold_results_cat = []
fold_results_cat_cal = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, strata), start=1):
    start_time = time.perf_counter()
    print(f"\n========== FOLD {fold} ==========")

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y_series.iloc[train_idx]
    y_test = y_series.iloc[test_idx]

    # ── Feature typing & missing value handling ────────────────────────────────
    # Separate numeric/bool features from categoricals; fill missing cat values
    # before passing to the OHE preprocessor
    num_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat_features = [c for c in X_train.columns if c not in num_features]

    X_train = fill_cat_missing(X_train, cat_features)
    X_test = fill_cat_missing(X_test, cat_features)

    # ── Class weighting ────────────────────────────────────────────────────────
    # Compute balanced class weights 
    # the penalty — helps counter the heavy class imbalance in churn labels
    class_labels = np.unique(y_train)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=class_labels,
        y=y_train,
    )
    class_weight_dict = {
        label: weight
        for label, weight in zip(class_labels, class_weights)
    }

    # ── Preprocessing pipeline ─────────────────────────────────────────────────
    # Numeric features passed through as-is; categoricals one-hot encoded
    # Unknown categories at test time are silently ignored
    preprocess = ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ],
        remainder="drop",
    )

    # ══════════════════════════════════════════════════════════════════════════
    # BLOCK 1 — Random Forest (no calibration)
    # Train RF on the full training fold and generate raw softmax probabilities.
    # These serve as the uncalibrated baseline for comparison.
    # ══════════════════════════════════════════════════════════════════════════
    rf = Pipeline(
        steps=[
            ("prep", preprocess),
            ("rf", RandomForestClassifier(
                random_state=42,
                n_jobs=-1,
                class_weight=class_weight_dict,
            )),
        ]
    )

    rf.fit(X_train, y_train)

    proba_test = rf.predict_proba(X_test)
    fold_classes = rf.named_steps["rf"].classes_

    y_pred = fold_classes[np.argmax(proba_test, axis=1)]
    confidence = np.max(proba_test, axis=1)

    fold_df_uncal = (
        pd.DataFrame(
            proba_test,
            index=X_test.index,
            columns=[f"p_{c}" for c in fold_classes],
        )
        .assign(
            y_true=y_test.to_numpy(),
            y_pred=y_pred,
            confidence=confidence,
            model="rf_uncal",
            fold=fold,
        )
    )
    fold_results_uncal.append(fold_df_uncal)

    # ══════════════════════════════════════════════════════════════════════════
    # BLOCK 2 — Random Forest + Dirichlet calibration
    # Train RF on a 70% sub-split, then fit a FullDirichletCalibrator on the
    # held-out 30% calibration set. The minority group in the calibration set is oversampled to be
    # class-balanced before fitting, to prevent Dirichlet collapsing predictions into the majority class.
    # ══════════════════════════════════════════════════════════════════════════

    # Split training fold into fit / calibration subsets, stratified on
    # treatment × outcome to preserve label balance in both halves
    t_train = t_s.iloc[train_idx]
    strata_train = t_train + "_" + y_train.astype(str)

    X_fit, X_calib, y_fit, y_calib, t_fit, t_calib = train_test_split(
        X_train,
        y_train,
        t_train,
        test_size=0.3,
        stratify=strata_train,
        random_state=42,
    )

    rf.fit(X_fit, y_fit)
    cal_classes = rf.named_steps["rf"].classes_

    # Get calibration probabilities and encode labels as integer codes
    p_cal = rf.predict_proba(X_calib)
    class_to_idx = {c: i for i, c in enumerate(cal_classes)}

    y_calib_codes = (
        pd.Series(y_calib, index=X_calib.index)
          .map(class_to_idx)
          .to_numpy()
    )

    # Oversample minority classes in the calibration set so Dirichlet sees a
    # balanced label distribution — avoids majority-class collapse
    temp_df = pd.DataFrame(p_cal)
    temp_df['target'] = y_calib_codes

    max_class_count = pd.Series(y_calib_codes).value_counts().max()

    balanced_df = pd.concat([
        resample(temp_df[temp_df['target'] == c],
                 replace=True,
                 n_samples=max_class_count,
                 random_state=42)
        for c in np.unique(y_calib_codes)
    ])

    p_cal_balanced = balanced_df.drop('target', axis=1).to_numpy()
    y_cal_balanced = balanced_df['target'].to_numpy()

    # Fit Dirichlet calibrator; watchdog logs progress every 60s in case of
    # slow convergence with fmin_l_bfgs_b
    print(f"[Dirichlet:RF fold={fold}] X shape={p_cal_balanced.shape} y shape={y_cal_balanced.shape}")
    print(f"[Dirichlet:RF fold={fold}] y counts: {pd.Series(y_cal_balanced).value_counts().to_dict()}")
    stop_evt, t0 = _start_fit_watchdog(label=f"RF fold={fold}", every_s=600)
    try:
        cal = FullDirichletCalibrator(optimizer="fmin_l_bfgs_b")
        cal.fit(p_cal_balanced, y_cal_balanced)
    finally:
        _stop_fit_watchdog(stop_evt, t0, label=f"RF fold={fold}")

    # Apply calibrator to raw test probabilities
    p_test_raw = rf.predict_proba(X_test)
    proba_test_cal = cal.predict_proba(p_test_raw)

    y_pred_cal = cal_classes[np.argmax(proba_test_cal, axis=1)]
    confidence_cal = np.max(proba_test_cal, axis=1)

    print("RF cal y_pred dist:\n", pd.Series(y_pred_cal).value_counts(normalize=True))
    print("RF mean proba (cal):", proba_test_cal.mean(axis=0))

    fold_df_cal = (
        pd.DataFrame(
            proba_test_cal,
            index=X_test.index,
            columns=[f"p_{c}" for c in cal_classes],
        )
        .assign(
            y_true=y_test.to_numpy(),
            y_pred=y_pred_cal,
            confidence=confidence_cal,
            model="rf_calibrated_dirichlet",
            fold=fold,
        )
    )
    fold_results_cal.append(fold_df_cal)

    # ══════════════════════════════════════════════════════════════════════════
    # BLOCK 3 — CatBoost (no calibration)
    # Train CatBoost on the full training fold using native categorical feature
    # ══════════════════════════════════════════════════════════════════════════
    class_names = list(class_labels)

    cat = CatBoostClassifier(
        verbose=100,
        random_seed=42,
        class_names=class_names,
        class_weights=[class_weight_dict[c] for c in class_labels],
        loss_function="MultiClass",
        thread_count=-1,
    )

    cat.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    proba_cat_test = cat.predict_proba(X_test)
    cat_classes = cat.classes_

    # Re-align CatBoost class probabilities to the global all_classes ordering
    # since CatBoost may return classes in a different order per fold
    proba_cat_aligned = np.zeros((len(test_idx), len(all_classes)), dtype=float)
    for j, c in enumerate(cat_classes):
        proba_cat_aligned[:, np.where(all_classes == c)[0][0]] = proba_cat_test[:, j]

    y_pred_cat = cat_classes[np.argmax(proba_cat_test, axis=1)]
    confidence_cat = np.max(proba_cat_test, axis=1)

    fold_df_cat = (
        pd.DataFrame(
            proba_cat_aligned,
            index=X_test.index,
            columns=[f"p_{c}" for c in all_classes],
        )
        .assign(
            y_true=y_test.to_numpy(),
            y_pred=y_pred_cat,
            confidence=confidence_cat,
            model="catboost_uncal",
            fold=fold,
        )
    )
    fold_results_cat.append(fold_df_cat)

    # ══════════════════════════════════════════════════════════════════════════
    # BLOCK 4 — CatBoost + Dirichlet calibration
    # Mirrors the RF calibration approach: re-train on the 70% fit split,
    # calibrate on the oversampled 30% hold-out, then apply to test set.
    # ══════════════════════════════════════════════════════════════════════════

    # Re-train CatBoost on the fit subset (same split used for RF calibration)
    cat.fit(X_fit, y_fit, cat_features=cat_features)
    cal_classes = cat.classes_

    # Encode calibration labels as integer codes aligned to CatBoost class order
    proba_calib = cat.predict_proba(X_calib)
    y_calib_codes = pd.Categorical(y_calib, categories=cal_classes).codes

    # Oversample calibration set to balanced class counts (same logic as RF above)
    temp_df = pd.DataFrame(proba_calib)
    temp_df['target'] = y_calib_codes

    max_class_count = pd.Series(y_calib_codes).value_counts().max()

    balanced_df = pd.concat([
        resample(temp_df[temp_df['target'] == c],
                 replace=True,
                 n_samples=max_class_count,
                 random_state=42)
        for c in np.unique(y_calib_codes)
    ])

    proba_calib_bal = balanced_df.drop('target', axis=1).to_numpy()
    y_cal_bal = balanced_df['target'].to_numpy()

    # Fit Dirichlet calibrator; watchdog set to 600s given CatBoost's longer
    # calibration convergence time
    print(f"[Dirichlet:CAT fold={fold}] X shape={proba_calib_bal.shape} y shape={y_cal_bal.shape}")
    print(f"[Dirichlet:CAT fold={fold}] y counts: {pd.Series(y_cal_bal).value_counts().to_dict()}")
    stop_evt, t0 = _start_fit_watchdog(label=f"CAT fold={fold}", every_s=600)
    try:
        cal = FullDirichletCalibrator(optimizer="fmin_l_bfgs_b")
        cal.fit(proba_calib_bal, y_cal_bal)
    finally:
        _stop_fit_watchdog(stop_evt, t0, label=f"CAT fold={fold}")

    # Apply calibrator to raw CatBoost test probabilities
    proba_test_raw = cat.predict_proba(X_test)
    proba_test_cal = cal.predict_proba(proba_test_raw)

    y_pred_cal = cal_classes[np.argmax(proba_test_cal, axis=1)]
    confidence_cal = np.max(proba_test_cal, axis=1)

    print("Cat cal y_pred dist:\n", pd.Series(y_pred_cal).value_counts(normalize=True))
    print("Cat mean proba (cal):", proba_test_cal.mean(axis=0))

    fold_df_cat_cal = (
        pd.DataFrame(
            proba_test_cal,
            index=X_test.index,
            columns=[f"p_{c}" for c in cal_classes],
        )
        .assign(
            y_true=y_test.to_numpy(),
            y_pred=y_pred_cal,
            confidence=confidence_cal,
            model="catboost_calibrated_dirichlet",
            fold=fold,
        )
    )
    fold_results_cat_cal.append(fold_df_cat_cal)

    elapsed = time.perf_counter() - start_time
    print(f"FOLD {fold} took {elapsed:.2f} seconds")

# ── Combine fold results & restore original index order ───────────────────────
# concatenating out-of-order fold predictions
df_uncalibrated_rf = pd.concat(fold_results_uncal, axis=0).loc[X.index]
df_calibrated_rf_dirichlet = pd.concat(fold_results_cal, axis=0).loc[X.index]
df_uncalibrated_catboost = pd.concat(fold_results_cat, axis=0).loc[X.index]
df_calibrated_catboost_dirichlet = pd.concat(fold_results_cat_cal, axis=0).loc[X.index]


========== FOLD 1 ==========
[Dirichlet:RF fold=1] X shape=(397968, 16) y shape=(397968,)
[Dirichlet:RF fold=1] y counts: {0: 24873, 1: 24873, 2: 24873, 3: 24873, 4: 24873, 5: 24873, 6: 24873, 7: 24873, 8: 24873, 9: 24873, 10: 24873, 11: 24873, 12: 24873, 13: 24873, 14: 24873, 15: 24873}
[Dirichlet:RF fold=1] still fitting (elapsed=1.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=2.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=3.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=4.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=5.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=6.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=7.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=8.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=9.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=10.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=11.0 min)
[Dirichlet:RF fold=1] still fitting (elapsed=12.0 min)
[Dirichlet:RF fold=1] fit finished (elapsed=12.18 min)
RF c

In [16]:
df_calibrated_rf_dirichlet.head()

,p_no_reactivated_0,p_no_reactivated_1,p_no_reactivated_2,p_no_reactivated_3,p_no_reactivated_4,p_no_reactivated_5,p_no_reactivated_6,p_no_reactivated_7,p_reactivated_0,p_reactivated_1,p_reactivated_2,p_reactivated_3,p_reactivated_4,p_reactivated_5,p_reactivated_6,p_reactivated_7,y_true,y_pred,confidence,model,fold
1,0.069236,0.061729,0.075334,0.067664,0.065025,0.067421,0.068159,0.072624,0.062412,4.051742e-02,0.069941,0.052872,1.128502e-01,1.922799e-105,0.082909,3.130568e-02,no_reactivated_7,reactivated_4,0.112850,rf_calibrated_dirichlet,4
2,0.062443,0.062159,0.066991,0.065601,0.064430,0.054776,0.063136,0.062995,0.069694,5.887933e-74,0.090501,0.074400,3.000746e-30,1.049815e-01,0.085725,7.217000e-02,no_reactivated_7,reactivated_5,0.104982,rf_calibrated_dirichlet,2
3,0.079159,0.081002,0.077374,0.078856,0.076160,0.077441,0.078228,0.073431,0.102723,1.054253e-160,0.078997,0.099589,2.751807e-67,1.360191e-45,0.097039,3.912912e-85,no_reactivated_1,reactivated_0,0.102723,rf_calibrated_dirichlet,5
6,0.039113,0.040671,0.037911,0.031783,0.038131,0.038669,0.032922,0.040433,0.087428,1.995972e-01,0.066717,0.093409,8.687468e-132,6.833382e-02,0.105047,7.983468e-02,no_reactivated_5,reactivated_1,0.199597,rf_calibrated_dirichlet,3
7,0.064582,0.066967,0.068907,0.072221,0.062017,0.065279,0.067187,0.061934,0.096412,2.718943e-160,0.088892,0.086169,9.137512e-02,6.469097e-46,0.108057,2.220155e-84,no_reactivated_0,reactivated_6,0.108057,rf_calibrated_dirichlet,5


In [17]:
class_weight_dict

{'no_reactivated_0': np.float64(0.12719216249140625),
 'no_reactivated_1': np.float64(0.9093192204880572),
 'no_reactivated_2': np.float64(0.8785616095976005),
 'no_reactivated_3': np.float64(0.91136245786881),
 'no_reactivated_4': np.float64(0.899776023890785),
 'no_reactivated_5': np.float64(0.8760072271141386),
 'no_reactivated_6': np.float64(0.900390625),
 'no_reactivated_7': np.float64(0.9186666957052008),
 'reactivated_0': np.float64(6.060560344827586),
 'reactivated_1': np.float64(41.19287109375),
 'reactivated_2': np.float64(39.05694444444445),
 'reactivated_3': np.float64(35.9910409556314),
 'reactivated_4': np.float64(37.39494680851064),
 'reactivated_5': np.float64(36.23840206185567),
 'reactivated_6': np.float64(34.80321782178218),
 'reactivated_7': np.float64(40.24952290076336)}

In [18]:
df_uncalibrated_rf.head()

,p_no_reactivated_0,p_no_reactivated_1,p_no_reactivated_2,p_no_reactivated_3,p_no_reactivated_4,p_no_reactivated_5,p_no_reactivated_6,p_no_reactivated_7,p_reactivated_0,p_reactivated_1,p_reactivated_2,p_reactivated_3,p_reactivated_4,p_reactivated_5,p_reactivated_6,p_reactivated_7,y_true,y_pred,confidence,model,fold
1,0.48,0.05,0.04,0.21,0.02,0.04,0.06,0.09,0.01,0.00,0.0,0.0,0.0,0.0,0.00,0.0,no_reactivated_7,no_reactivated_0,0.48,rf_uncal,4
2,0.63,0.01,0.00,0.04,0.13,0.09,0.02,0.08,0.00,0.00,0.0,0.0,0.0,0.0,0.00,0.0,no_reactivated_7,no_reactivated_0,0.63,rf_uncal,2
3,0.66,0.06,0.05,0.05,0.06,0.04,0.05,0.02,0.01,0.00,0.0,0.0,0.0,0.0,0.00,0.0,no_reactivated_1,no_reactivated_0,0.66,rf_uncal,5
6,0.62,0.09,0.04,0.06,0.00,0.03,0.04,0.05,0.04,0.01,0.0,0.0,0.0,0.0,0.02,0.0,no_reactivated_5,no_reactivated_0,0.62,rf_uncal,3
7,0.46,0.05,0.08,0.03,0.08,0.05,0.15,0.08,0.01,0.01,0.0,0.0,0.0,0.0,0.00,0.0,no_reactivated_0,no_reactivated_0,0.46,rf_uncal,5


In [19]:
df_preds = pd.concat(
    [
        df_uncalibrated_rf,
        df_calibrated_rf_dirichlet,
         df_uncalibrated_catboost,
         df_calibrated_catboost_dirichlet
    ],
    ignore_index=True,
)

df_preds = df_preds.assign(correct=df_preds["y_pred"] == df_preds["y_true"])

In [20]:
# Saving DF for calibration and uplift analytics
def save_df_preds(df_preds, path: str = "Data/df_preds.csv") -> None:
    df_preds.to_csv(path, index=False)
save_df_preds(df_preds)